# AE2: Obtención y reconocimiento del dataset Goodreads Books

Primera etapa del proyecto integrador. Este notebook obtiene el dataset desde Hugging Face, lo carga en un DataFrame de Pandas, lo describe brevemente y genera automáticamente su resumen en `AE2/resultados/resumen_dataframe.txt`.

**Escenario hipotético.** Una librería planea rediseñar su exhibición en dos áreas estratégicas: *Lecturas Ágiles* (libros de volumen delgado o mediano) y *Grandes Clásicos y Sagas* (obras de gran extensión). Para decidir qué porcentaje de espacio asignar a cada área y qué editoriales e idiomas priorizar en las compras importadas, requiere un diagnóstico descriptivo de los libros destacados (calificación promedio ≥ 3.5) en Goodreads.

Flujo:
1. Configuración de rutas y parámetros del dataset.
2. Obtención del dataset con `HFDatasetLoader` (`src/hf_loader.py`).
3. Descripción básica: registros, dimensiones, columnas, tipos y valores nulos.
4. Qué representa cada fila y qué variables respaldan la necesidad.
5. Generación del resumen `.txt` con `Reportes` (`src/reportes.py`).

## 1. Configuración

In [1]:
import os
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Ruta dinámica hacia la raíz del repositorio (la carpeta que contiene 'src')
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from datasets import disable_progress_bars

from hf_loader import HFDatasetLoader
from reportes import Reportes

# Sin barras de progreso para que la salida guardada del notebook quede limpia
disable_progress_bars()

# Parámetros del dataset utilizado
DATASET_ID = "pfaha/goodreads-books"
DATASET_URL = f"https://huggingface.co/datasets/{DATASET_ID}"
CONFIG = "default"
SPLIT = "train"
# El dataset crece con nuevos checkpoints; se fija la revisión consultada para reproducir los resultados
REVISION = "43b3198cd7ec9c142d56181f1a09ed9e8c112610"
FECHA_CONSULTA = "2026-10-24"

RUTA_RESUMEN = ROOT / "AE2" / "resultados" / "resumen_dataframe.txt"

print(f"Raíz del proyecto: {ROOT.name}")
print(f"Dataset: {DATASET_ID} | config: {CONFIG} | split: {SPLIT} | revisión: {REVISION[:7]}")

Raíz del proyecto: RepoMachine_ProInteg3-main
Dataset: pfaha/goodreads-books | config: default | split: train | revisión: 43b3198


## 2. Obtención del dataset

Se descarga el split `train` completo (no se usa una muestra). La descarga es pública y no requiere token; los archivos quedan en la caché local de Hugging Face y no se guardan en el repositorio.

In [2]:
loader = HFDatasetLoader(
    dataset_name=DATASET_ID,
    config=CONFIG,
    split=SPLIT,
    revision=REVISION,
)
df = loader.load()

print("Dataset cargado en un DataFrame de Pandas.")
loader.info()

Dataset cargado en un DataFrame de Pandas.


{'dataset': 'pfaha/goodreads-books',
 'config': 'default',
 'split': 'train',
 'revision': '43b3198cd7ec9c142d56181f1a09ed9e8c112610',
 'filas': 500000,
 'columnas': 24}

## 3. Descripción básica

### 3.1 Algunos registros

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)
df.head(5)

,title,contributors,first_author,description,genres,series,format,pages,language,asin,isbn10,isbn13,publisher,publication_date,average_rating,ratings_count,reviews_count,work_id,catalog_guideline_violation,book_id,url,num_editions,has_more_editions,is_part_of_series
0,La doctrina secreta. Vol I. Cosmogen...,[{'name': 'Helena Petrovna Blavatsky...,Helena Petrovna Blavatsky,"Esta es una obra inteligente, origin...","[Occult, Spirituality, Nonfiction, P...","[{'name': 'The Secret Doctrine', 'po...",Paperback,319.0,Spanish; Castilian,950171103X,950171103X,9789501711035,Editorial Kier,2005-12-15,4.16,255.0,22.0,44066165.0,False,7282453,https://www.goodreads.com/book/show/...,124.0,True,True
1,Clement Marot and the Inflections of...,"[{'name': 'Robert Griffin', 'url': '...",Robert Griffin,This title is part of UC Press's Voi...,[],[],Hardcover,NaN,English,0520025865,0520025865,9780520025868,University of California Press,1974-01-01,0.00,0.0,0.0,1320719.0,False,1331224,https://www.goodreads.com/book/show/...,2.0,True,False
2,How to Market Legal Services,"[{'name': 'Robert W Denney', 'url': ...",Robert W Denney,None,[],[],Hardcover,278.0,English,0442219806,0442219806,9780442219802,Van Nostrand Reinhold Company,1984-09-01,0.00,0.0,0.0,3956063.0,False,3910678,https://www.goodreads.com/book/show/...,2.0,True,False
3,Personality and Intellectual Competence,"[{'name': 'Tomas Chamorro-Premuzic',...",Tomas Chamorro-Premuzic,This book provides a comprehensive s...,"[Psychology, Reference]",[],Hardcover,214.0,English,0805851364,0805851364,9780805851366,Psychology Press,2014-06-19,3.88,8.0,0.0,580945.0,False,2027361,https://www.goodreads.com/book/show/...,8.0,True,False
4,Studies in the Text History Of the L...,"[{'name': 'Ben Edwin Perry', 'url': ...",Ben Edwin Perry,"English, Greek",[],[],Paperback,272.0,English,0891305343,0891305343,9780891305347,Oxford University Press,1981-05-01,0.00,0.0,0.0,3997874.0,False,3952223,https://www.goodreads.com/book/show/...,4.0,True,False


### 3.2 Dimensiones

In [4]:
filas, columnas = df.shape
print(f"Filas: {filas:,} | Columnas: {columnas}")

Filas: 500,000 | Columnas: 24


### 3.3 Columnas y tipos de datos

In [5]:
tipos = pd.DataFrame({
    "columna": df.columns,
    "tipo_pandas": df.dtypes.astype(str).values,
    "ejemplo": [repr(df[c].dropna().iloc[0])[:50] if df[c].notna().any() else None for c in df.columns],
})
tipos

,columna,tipo_pandas,ejemplo
0,title,object,'La doctrina secreta. Vol I. Cosmoge...
1,contributors,object,array([{'name': 'Helena Petrovna Bla...
2,first_author,object,'Helena Petrovna Blavatsky'
3,description,object,"'Esta es una obra inteligente, origi..."
4,genres,object,"array(['Occult', 'Spirituality', 'No..."
5,series,object,array([{'name': 'The Secret Doctrine...
6,format,object,'Paperback'
7,pages,float64,np.float64(319.0)
8,language,object,'Spanish; Castilian'
9,asin,object,'950171103X'


### 3.4 Valores nulos por columna

Además de los nulos, se cuentan los valores vacíos (texto `''` o lista vacía), porque el scraping suele dejar campos vacíos en lugar de nulos.

In [6]:
reportes = Reportes()
calidad = pd.DataFrame({
    "nulos": df.isna().sum(),
    "vacios": [reportes._contar_vacios(df[c]) for c in df.columns],
})
calidad["porcentaje_nulos_o_vacios"] = ((calidad["nulos"] + calidad["vacios"]) / len(df) * 100).round(2)
calidad

,nulos,vacios,porcentaje_nulos_o_vacios
title,0,0,0.00
contributors,0,160,0.03
first_author,160,0,0.03
description,65575,14,13.12
genres,0,307620,61.52
series,0,437717,87.54
format,5107,2258,1.47
pages,38992,0,7.80
language,19787,0,3.96
asin,2909,0,0.58


## 4. Qué representa cada fila y qué variables respaldan la necesidad

**Unidad de análisis.** Cada fila es un libro (edición) de Goodreads identificado por `book_id`, obtenido por muestreo aleatorio de identificadores. Los campos bibliográficos (`pages`, `publisher`, `language`, `format`) describen esa edición, mientras que `average_rating`, `ratings_count` y `reviews_count` se agregan a nivel de la obra (todas sus ediciones), según la ficha del dataset.

Observaciones de la revisión:
- `pages`, `ratings_count`, `reviews_count`, `work_id` y `num_editions` figuran como `int64` en la ficha, pero Pandas las carga como `float64` porque contienen nulos.
- En los registros revisados, una obra sin calificaciones (`ratings_count = 0`) aparece con `average_rating = 0.00`. Ese 0 indica ausencia de calificaciones, no una calificación baja; no afecta al filtro ≥ 3.5.
- `genres` y `series` usan listas vacías en lugar de nulos.

**Variables que respaldan la necesidad:**

| Variable | Uso en la necesidad |
|---|---|
| `pages` | Clasificar los libros en *Lecturas Ágiles* o *Grandes Clásicos y Sagas* según su extensión. |
| `average_rating` | Delimitar los libros destacados (≥ 3.5) y comparar la valoración entre grupos. |
| `ratings_count`, `reviews_count` | Medir la participación de los lectores en cada grupo. |
| `publisher` | Identificar las editoriales que concentran más títulos destacados. |
| `language` | Identificar los idiomas que conviene priorizar en las compras importadas. |
| `is_part_of_series`, `series` | Caracterizar las sagas dentro del área de grandes obras. |
| `format`, `genres` | Variables de apoyo para describir cada área. |

### Dimensionamiento preliminar

El conteo siguiente solo dimensiona la población de interés; no es el análisis de la etapa 2. Los cortes son **provisionales**: un número de páginas válido entre 1 y 5,000 (para excluir valores anómalos como 0 o cifras de millones) y un umbral de 350 páginas entre las dos áreas, cercano al tercer cuartil observado. Ambos se validarán en la siguiente etapa.

In [7]:
PAGINAS_MIN, PAGINAS_MAX = 1, 5000   # rango provisional de páginas válidas
UMBRAL_AGILES = 350                  # umbral provisional entre las dos áreas

destacados = df[df["average_rating"] >= 3.5]
validos = destacados[destacados["pages"].between(PAGINAS_MIN, PAGINAS_MAX)]
area = pd.cut(
    validos["pages"],
    bins=[PAGINAS_MIN - 1, UMBRAL_AGILES, PAGINAS_MAX],
    labels=["Lecturas Ágiles", "Grandes Clásicos y Sagas"],
)

print(f"Libros con calificación >= 3.5: {len(destacados):,} de {len(df):,}")
print(f"De ellos, con páginas entre {PAGINAS_MIN} y {PAGINAS_MAX}: {len(validos):,}")
informado = lambda serie: serie.fillna("").str.strip().ne("").sum()
print(f"Con editorial informada: {informado(validos['publisher']):,}")
print(f"Con idioma informado: {informado(validos['language']):,}\n")
area.value_counts().rename("libros").to_frame().assign(
    porcentaje=lambda t: (t["libros"] / t["libros"].sum() * 100).round(1)
)

Libros con calificación >= 3.5: 276,152 de 500,000
De ellos, con páginas entre 1 y 5000: 235,104
Con editorial informada: 228,189
Con idioma informado: 229,528



,libros,porcentaje
pages,,
Lecturas Ágiles,175788,74.8
Grandes Clásicos y Sagas,59316,25.2


## 5. Generación automática del resumen `.txt`

El resumen se genera a partir del mismo DataFrame `df` revisado arriba. Se guarda en UTF-8 en una ruta relativa del proyecto; la carpeta se crea si no existe y el archivo se sobrescribe en cada ejecución.

In [8]:
metadatos = {
    "Dataset": DATASET_ID,
    "Enlace": DATASET_URL,
    "Configuración": CONFIG,
    "Split": SPLIT,
    "Revisión": REVISION,
    "Fecha de consulta": FECHA_CONSULTA,
    "Muestra": "No; se resume el split train completo de la revisión indicada",
}

ruta_resumen = reportes.guardar_resumen_dataframe(df, RUTA_RESUMEN, metadatos)
print(f"Resumen generado en: {ruta_resumen.relative_to(ROOT).as_posix()}\n")

# Verificación: se vuelve a leer el archivo generado
print(ruta_resumen.read_text(encoding="utf-8"))

Resumen generado en: AE2/resultados/resumen_dataframe.txt

RESUMEN DEL DATAFRAME
Dataset: pfaha/goodreads-books
Enlace: https://huggingface.co/datasets/pfaha/goodreads-books
Configuración: default
Split: train
Revisión: 43b3198cd7ec9c142d56181f1a09ed9e8c112610
Fecha de consulta: 2026-10-24
Muestra: No; se resume el split train completo de la revisión indicada

Filas: 500000
Columnas: 24

Columnas y tipos:
- title: object
- contributors: object
- first_author: object
- description: object
- genres: object
- series: object
- format: object
- pages: float64
- language: object
- asin: object
- isbn10: object
- isbn13: object
- publisher: object
- publication_date: object
- average_rating: float64
- ratings_count: float64
- reviews_count: float64
- work_id: float64
- catalog_guideline_violation: bool
- book_id: object
- url: object
- num_editions: float64
- has_more_editions: bool
- is_part_of_series: bool

Valores nulos por columna:
- title: 0
- contributors: 0
- first_author: 160
- descri